# **Proyecto 06: Detector de Spam en Comentarios de YouTube con Red Neuronal**

---

Actualizamos el detector de spam anterior para que utilice **redes neuronales** en lugar del bosque aleatorio.

El dataset contiene aproximadamente **2000 comentarios** de 5 vídeos diferentes de YouTube, donde la mitad son spam y la otra mitad no.

**Diferencias respecto a la versión anterior (Random Forest):**
- Usamos **TF-IDF** con 2000 palabras (antes 1000 con CountVectorizer)
- Usamos una **red neuronal superficial** en lugar del bosque aleatorio
- Validación cruzada con **StratifiedKFold** (5 pliegues)
- Precisión esperada: ~95%

---
## **Parte 1: Importación de librerías**

In [15]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [16]:
import pandas as pd
import numpy as np
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import StratifiedKFold

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


---
## **Parte 2: Carga del dataset**

Cargamos los **5 archivos CSV** correspondientes a los comentarios de 5 vídeos de YouTube:
- Psy, Katy Perry, LMFAO, Eminem y Shakira

Los apilamos en un único DataFrame y los mezclamos aleatoriamente con `sample(frac=1)`.

In [17]:
import os
print(os.getcwd())

C:\Users\usuario\Red neuronal IA


In [18]:
d = pd.read_csv('Youtube_Combined_Cleaned.csv')

d = d.sample(frac=1)
d = d.reset_index(drop=True)

print(f'Total de comentarios: {len(d)}')
print(f'  Spam    (CLASS=1): {len(d.query("CLASS == 1"))}')
print(f'  No spam (CLASS=0): {len(d.query("CLASS == 0"))}')
print(d.head())

Total de comentarios: 1953
  Spam    (CLASS=1): 1003
  No spam (CLASS=0): 950
                                    COMMENT_ID        AUTHOR  \
0  LneaDw26bFsILb1-yJtTNbl7IIOeXIvJxxyU3o-EtR4   Yuliya Meow   
1          z13iupjoosrpzlm5v04cf32q4oqizvsbkdo    Jack Other   
2            z131i5aanxneixud422egfx4rsmugbpm1        NoName   
3          z13nw3lhgt2nf5wwe04cdlx5iyaydznrve0  Wert Walleet   
4        z12rtnfbgv2jwdj4q04cefeaqxevh5ogpwg0k   Роза Герман   

                  DATE                                            CONTENT  \
0                  NaN  HEY GUYS!!! ❤❤❤❤❤❤❤  BEFORE YOU IGNORE ME, PLE...   
1  2014-08-18 18:05:37  http://www.amazon.com/Knight-Dawn-cursed-Danie...   
2                  NaN  870,000,000 views...566,000 comments...oh my l...   
3  2014-11-08 09:15:22  This song is great there are 2,127,315,950 vie...   
4                  NaN                               Very good! Like! :D﻿   

   CLASS  
0      1  
1      1  
2      0  
3      0  
4      0  


---
## **Parte 3: Validación cruzada con StratifiedKFold**

Dividimos los datos en **5 pliegues (folds)**. Cada fold usará un 80% para entrenamiento y un 20% para prueba, garantizando que la proporción de clases sea similar en cada partición.

In [19]:
# Creamos el objeto StratifiedKFold con 5 divisiones
kfold = StratifiedKFold(n_splits=5)

# Generamos los índices de cada split
splits = kfold.split(d, d['CLASS'])

# Comprobamos que los splits no se superponen
print('🔀 Índices de test para cada fold:')
for i, (train, test) in enumerate(kfold.split(d, d['CLASS'])):
    print(f'\nSplit {i+1} (primeros 20 índices): {test[:20]}')

🔀 Índices de test para cada fold:

Split 1 (primeros 20 índices): [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]

Split 2 (primeros 20 índices): [362 363 364 369 370 374 376 377 378 379 382 383 386 387 388 389 391 392
 393 395]

Split 3 (primeros 20 índices): [759 761 763 764 771 772 774 776 778 779 782 784 785 786 790 792 793 794
 796 800]

Split 4 (primeros 20 índices): [1170 1171 1173 1175 1176 1177 1178 1179 1180 1181 1183 1184 1185 1186
 1187 1188 1189 1190 1191 1192]

Split 5 (primeros 20 índices): [1550 1551 1554 1558 1559 1560 1561 1562 1563 1564 1565 1567 1568 1571
 1573 1577 1578 1579 1581 1582]


---
## **Parte 4: Función `train_and_test`**

Esta función recibe los índices de train y test de cada fold y:

1. **Extrae** los comentarios de train y test
2. **Construye** el Tokenizer (Bag of Words con TF-IDF, 2000 palabras)
3. **Normaliza** los datos entre -1 y 1
4. **Construye** la red neuronal:
   - Capa densa: 512 neuronas, activación ReLU
   - Dropout: 0.5 (previene sobreajuste)
   - Capa densa de salida: 2 neuronas, activación Softmax
5. **Entrena** el modelo (10 épocas, batch_size=16)
6. **Evalúa** y devuelve la precisión

In [20]:
def train_and_test(train_idx, test_idx):

    # 1. Extraemos los comentarios
    train_content = d['CONTENT'].iloc[train_idx]
    test_content  = d['CONTENT'].iloc[test_idx]

    # 2. Tokenizador TF-IDF (2000 palabras)
    tokenizer = Tokenizer(num_words=2000)
    tokenizer.fit_on_texts(train_content)

    d_train_inputs = tokenizer.texts_to_matrix(train_content, mode='tfidf')
    d_test_inputs  = tokenizer.texts_to_matrix(test_content,  mode='tfidf')

    # 3. Normalización entre -1 y 1
    d_train_inputs = d_train_inputs / np.amax(np.absolute(d_train_inputs))
    d_test_inputs  = d_test_inputs  / np.amax(np.absolute(d_test_inputs))

    d_train_inputs = d_train_inputs - np.mean(d_train_inputs)
    d_test_inputs  = d_test_inputs  - np.mean(d_test_inputs)

    # 4. Etiquetas one-hot
    d_train_outputs = to_categorical(d['CLASS'].iloc[train_idx])
    d_test_outputs  = to_categorical(d['CLASS'].iloc[test_idx])

    # 5. Red neuronal — MEJORA 2: capa extra de 256 neuronas
    model = Sequential()

    model.add(Dense(512, input_shape=(2000,)))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))

    # ← Nueva capa intermedia
    model.add(Dense(256))
    model.add(Activation('relu'))
    model.add(Dropout(0.3))

    model.add(Dense(2))
    model.add(Activation('softmax'))

    # 6. Compilación
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adamax',
        metrics=['accuracy']
    )

    # 7. MEJORA 1: Early Stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    # 8. Entrenamiento con validación y early stopping
    model.fit(
        d_train_inputs,
        d_train_outputs,
        epochs=10,
        batch_size=16,
        validation_split=0.1,   # 10% de train como validación interna
        callbacks=[early_stop],
        verbose=1
    )

    # 9. Evaluación
    scores = model.evaluate(d_test_inputs, d_test_outputs, verbose=0)

    return scores, model, tokenizer   # ← Devolvemos también modelo y tokenizer

---
## **Parte 5: Ejecución de la validación cruzada**

Ejecutamos la función `train_and_test` para cada uno de los 5 folds y recopilamos las precisiones.

In [21]:
kfold = StratifiedKFold(n_splits=5)
splits = kfold.split(d, d['CLASS'])

cvscores = []
mejor_accuracy = 0
mejor_modelo = None
mejor_tokenizer = None

for i, (train_idx, test_idx) in enumerate(splits):
    print('\n' + '='*40)
    print('  FOLD ' + str(i+1) + ' / 5')
    print('='*40)

    scores, model, tokenizer = train_and_test(train_idx, test_idx)
    accuracy = scores[1] * 100
    cvscores.append(accuracy)
    print('\n  Precision Fold ' + str(i+1) + ': ' + str(round(accuracy, 2)) + '%')

    # MEJORA 3: guardamos el fold con mejor precisión
    if accuracy > mejor_accuracy:
        mejor_accuracy = accuracy
        mejor_modelo = model
        mejor_tokenizer = tokenizer

# Guardamos el mejor modelo y tokenizer para Streamlit
mejor_modelo.save('modelo_spam.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(mejor_tokenizer, f)

print('\n✅ Mejor modelo guardado: ' + str(round(mejor_accuracy, 2)) + '%')


  FOLD 1 / 5
Epoch 1/10


C:\Users\usuario\miniconda3\envs\cronograma\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


88/88 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.6584 - loss: 0.6392 - val_accuracy: 0.8153 - val_loss: 0.5700
Epoch 2/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.8932 - loss: 0.4106 - val_accuracy: 0.8917 - val_loss: 0.3304
Epoch 3/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9331 - loss: 0.2152 - val_accuracy: 0.9299 - val_loss: 0.2378
Epoch 4/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9552 - loss: 0.1450 - val_accuracy: 0.9363 - val_loss: 0.2032
Epoch 5/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9616 - loss: 0.1177 - val_accuracy: 0.9363 - val_loss: 0.1865
Epoch 6/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9694 - loss: 0.1013 - val_accuracy: 0.9363 - val_loss: 0.1815
Epoch 7/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9737 - loss: 0.0860 - val_accuracy: 0.9363 - val_loss: 0.1694
Epoch 8/10
88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9758 - loss: 0.0783 - val_accuracy: 0.9299 - val_loss: 0.

---
## **Parte 6: Resultado final**

Calculamos la **media** y la **desviación estándar** de las precisiones obtenidas en los 5 folds.

In [13]:
print('🎯 Precisión media de la validación cruzada:')
print('%.2f%% (+/- %.2f%%)' % (np.mean(cvscores), np.std(cvscores)))

# Resultado esperado según el libro: ~95.09% (+/- 1.72%)

🎯 Precisión media de la validación cruzada:
94.62% (+/- 0.98%)


---
## **Resumen de la arquitectura**

```
Entrada: 2000 palabras (TF-IDF, normalizado entre -1 y 1)
    │
    ▼
┌─────────────────────┐
│  Dense(512)  + ReLU │  ← Capa oculta
└─────────────────────┘
    │
    ▼
┌─────────────────────┐
│    Dropout(0.5)     │  ← Regularización
└─────────────────────┘
    │
    ▼
┌─────────────────────┐
│ Dense(2) + Softmax  │  ← Capa de salida
└─────────────────────┘
    │
    ▼
Salida: [P(no spam), P(spam)]
```

| Hiperparámetro | Valor |
|---|---|
| Vocabulario | 2000 palabras |
| Representación | TF-IDF |
| Neuronas capa oculta | 512 |
| Activación oculta | ReLU |
| Dropout | 0.5 |
| Neuronas salida | 2 |
| Activación salida | Softmax |
| Función de pérdida | Categorical Crossentropy |
| Optimizador | Adamax |
| Épocas | 10 |
| Batch size | 16 |
| Validación | StratifiedKFold (k=5) |
| Precisión esperada | ~95% |